# Run Real Video VAE Latent Probe

语义 Stage：`real_video_vae_latent_probe` 过渡准备。这个 notebook 只负责在 Colab 会话中调度 runtime workspace、session model、runner、checker 与 package wrapper；正式执行逻辑由 `paper_workflow.notebook_utils.real_video_vae_latent_probe_workflow` 委托仓库模块完成。

### 00 Runtime Identity And User Config (`00_runtime_identity_and_user_config`)

作用：声明本次 formal workflow 的运行身份、输入数据集、结果 family 和本地 runtime 路径。

参数说明：`RUN_MODE` 与 `RUNTIME_PROFILE` 控制运行模式；`FAMILY_ID` 必须替换为本次正式结果包身份；`PROCESSED_DATASET_KEY` 必须来自上游 processed dataset handoff；`MODEL_ID` 指定本次 session-only VAE 模型；`REQUIRE_FORMAL_PASS=True` 是正式门禁，不能在 formal run 中关闭。

In [ ]:
from pathlib import Path
import os
import subprocess
import sys

from paper_workflow.notebook_utils import real_video_vae_latent_probe_workflow as probe_workflow

RUN_MODE = 'formal'
RUNTIME_PROFILE = 'formal'
WORKFLOW_KEY = 'real_video_vae_latent_probe_completion_formal'
STEP_KEY = 'step02_run_real_video_vae_latent_probe'
FAMILY_ID = 'real_video_vae_latent_probe__formal__davis2017_trainval480p__utc_time__short_commit'
PROCESSED_DATASET_KEY = 'real_video_vae_latent_probe__davis2017_trainval480p__256x256__32f__8fps__freeze001'
REPO_ROOT = Path.cwd().resolve()
DRIVE_ROOT = Path('/content/drive/MyDrive')
LOCAL_RUNTIME_ROOT = Path('/content/TSTW_runtime')
PROCESSED_DATASET_ROOT = DRIVE_ROOT / 'TSTW' / 'datasets' / 'processed' / PROCESSED_DATASET_KEY
LOCAL_DATASET_ROOT = LOCAL_RUNTIME_ROOT / 'datasets' / PROCESSED_DATASET_KEY
LOCAL_MODEL_ROOT = LOCAL_RUNTIME_ROOT / 'session_models' / 'autoencoder_kl'
FAMILY_ROOT = DRIVE_ROOT / 'TSTW' / 'results' / 'families' / FAMILY_ID
RUN_ROOT = LOCAL_RUNTIME_ROOT / 'runs' / 'real_video_vae_latent_probe_formal'
RUNTIME_CONFIG_PATH = RUN_ROOT / 'artifacts' / 'runtime_config.json'
SESSION_MODEL_MANIFEST_PATH = RUN_ROOT / 'artifacts' / 'session_model_manifest.json'
MODEL_ID = 'stabilityai/sd-vae-ft-mse'
REQUIRE_FORMAL_PASS = True

### 01 Mount Google Drive (`01_mount_google_drive`)

作用：挂载 Google Drive，读取 processed dataset 和 registry，并把最终 family package 写回 Drive。

In [ ]:
from google.colab import drive

drive.mount('/content/drive')

### 02 Read Processed Dataset Registry (`02_read_processed_dataset_registry`)

作用：定位 processed dataset registry，并确认本次 `PROCESSED_DATASET_KEY` 是下游运行要使用的正式输入身份。

In [ ]:
dataset_registry_path = DRIVE_ROOT / 'TSTW' / 'datasets' / 'registry' / 'dataset_registry.json'
print({'dataset_registry_path': str(dataset_registry_path), 'processed_dataset_key': PROCESSED_DATASET_KEY})

### 03 Prepare Local Runtime Workspace (`03_prepare_local_runtime_workspace`)

作用：调用 wrapper 准备 Colab 本地 runtime workspace，并把 Drive 中的 processed dataset 复制到 `/content/TSTW_runtime`。

参数说明：`copy_processed_dataset=True` 表示正式运行从本地副本读取，避免高频访问 Drive；`family_root` 是最终归档位置，`run_root` 是本次会话的临时执行根目录。

In [ ]:
runtime_workspace = probe_workflow.prepare_probe_runtime_workspace(
    processed_dataset_root=PROCESSED_DATASET_ROOT,
    local_dataset_root=LOCAL_DATASET_ROOT,
    family_root=FAMILY_ROOT,
    run_root=RUN_ROOT,
    copy_processed_dataset=True,
)
print(runtime_workspace)

### 04 Clone Or Update Repository (`04_clone_or_update_repository`)

作用：确认当前 notebook 正在仓库根目录中运行；如果在 Colab 中另行 clone，本 cell 应在 clone 后更新 `REPO_ROOT`，并打印最终仓库根路径。

In [ ]:
print({'repo_root': str(REPO_ROOT)})

### 05 Install Runtime Dependencies (`05_install_runtime_dependencies`)

作用：安装本次 Colab runtime 需要的轻量依赖。

参数说明：依赖列表应只包含 notebook session 需要的包；这里安装 Hugging Face 下载、Diffusers VAE、mp4 runtime、LPIPS 与图像质量指标依赖。正式算法或 checker 逻辑不得写在安装 cell 中。

In [ ]:
subprocess.run([
    sys.executable,
    '-m',
    'pip',
    'install',
    '-q',
    'huggingface_hub',
    'diffusers',
    'accelerate',
    'transformers',
    'safetensors',
    'imageio',
    'imageio-ffmpeg',
    'lpips',
    'scikit-image',
], check=True)

### 06 Prepare Session Model (`06_prepare_session_model`)

作用：调用 wrapper 下载并登记 session-only VAE 模型，模型权重只保存在 Colab 本地会话，不写入 Drive。

参数说明：`MODEL_ID` 控制模型来源；`LOCAL_MODEL_ROOT` 是本地权重目录；`SESSION_MODEL_MANIFEST_PATH` 是可归档的模型会话 manifest。

In [ ]:
session_model_manifest = probe_workflow.prepare_probe_session_model(
    model_id=MODEL_ID,
    local_model_root=LOCAL_MODEL_ROOT,
    session_manifest_path=SESSION_MODEL_MANIFEST_PATH,
)

### 07 Write Runtime Config (`07_write_runtime_config`)

作用：写出 runner 使用的 runtime config，记录本地数据集路径、Drive processed dataset 路径、session model 路径和 formal gate 配置。

参数说明：`execution_environment='colab'` 标记运行环境；`require_formal_pass_criteria=REQUIRE_FORMAL_PASS` 继承 formal 门禁；模型和数据路径必须来自前面已准备好的 workspace。

In [ ]:
runtime_config_handoff = probe_workflow.write_probe_runtime_config(
    runtime_config_path=RUNTIME_CONFIG_PATH,
    execution_environment='colab',
    processed_dataset_key=PROCESSED_DATASET_KEY,
    local_dataset_root=LOCAL_DATASET_ROOT,
    processed_dataset_root=PROCESSED_DATASET_ROOT,
    vae_model_local_path=LOCAL_MODEL_ROOT,
    require_formal_pass_criteria=REQUIRE_FORMAL_PASS,
)
print(runtime_config_handoff)

### 08 Check GPU And Runtime (`08_check_gpu_and_runtime`)

作用：通过 wrapper 执行 runtime preflight，检查数据集、本地模型目录和 formal run 环境是否满足要求。

In [ ]:
runtime_preflight = probe_workflow.run_probe_runtime_preflight(
    run_mode=RUN_MODE,
    local_dataset_root=LOCAL_DATASET_ROOT,
    local_model_dirs=[LOCAL_MODEL_ROOT],
)
print(runtime_preflight)

### 09 Verify Repository Contract (`09_verify_repository_contract`)

作用：运行项目契约校验和全量 audit，确保 notebook 执行前仓库仍处于受治理状态。

参数说明：`REPO_ROOT` 同时作为 subprocess 的 `cwd` 与 `PYTHONPATH` 前缀，避免 Colab 或脚本入口找不到 `main`、`experiments`、`tools` 模块。

In [ ]:
repo_env = dict(os.environ)
previous_pythonpath = repo_env.get('PYTHONPATH', '')
repo_env['PYTHONPATH'] = str(REPO_ROOT) if not previous_pythonpath else f'{REPO_ROOT}{os.pathsep}{previous_pythonpath}'
subprocess.run([sys.executable, 'tools/harness/validate_project_contract.py'], check=True, cwd=REPO_ROOT, env=repo_env)
subprocess.run([sys.executable, 'tools/harness/run_all_audits.py'], check=True, cwd=REPO_ROOT, env=repo_env)

### 10 Run Smoke Tests (`10_run_smoke_tests`)

作用：运行约束测试作为 formal runner 前的快速门禁；失败时应先修复仓库契约问题，再继续正式运行。

In [ ]:
subprocess.run([sys.executable, '-m', 'pytest', '-q', 'tests/constraints'], check=True, cwd=REPO_ROOT, env=repo_env)

### 11 Run Real Video VAE Latent Formal (`11_run_real_video_vae_latent_formal`)

作用：调用 `probe_workflow.run_probe_runner(...)` 运行正式 probe runner。notebook 不直接 import runner，也不在 cell 中实现 runner 逻辑。

In [ ]:
probe_workflow.run_probe_runner(
    run_root=RUN_ROOT,
    run_mode=RUN_MODE,
    runtime_profile=RUNTIME_PROFILE,
    runtime_config_path=RUNTIME_CONFIG_PATH,
)

### 12 Rebuild Tables And Reports (`12_rebuild_tables_and_reports`)

作用：从 runner 输出的 records 重建 tables、figures 和 reports，重建逻辑由 wrapper 委托 repository artifact builder 完成。

In [ ]:
probe_workflow.rebuild_probe_tables_and_reports(run_root=RUN_ROOT)

### 13 Check Real Video VAE Latent Outputs (`13_check_real_video_vae_latent_outputs`)

作用：执行正式输出检查，`formal_validation_summary['status']` 不通过时立即抛错，阻止未达标结果进入 package。

In [ ]:
formal_validation_summary = probe_workflow.check_probe_outputs(
    run_root=RUN_ROOT,
    construction_phase='real_video_vae_latent_probe',
    run_mode=RUN_MODE,
    require_formal_pass_criteria=REQUIRE_FORMAL_PASS,
)
if not formal_validation_summary['status']:
    raise RuntimeError(formal_validation_summary)

### 14 Package Family Results (`14_package_family_results`)

作用：调用 wrapper 打包 family result、写入 Drive archive，并追加 result/family registry handoff。

参数说明：`family_id`、`workflow_key`、`step_key` 必须与本次配置一致；`require_formal_pass_criteria` 必须使用 `REQUIRE_FORMAL_PASS`，避免 package 绕过 formal gate。

In [ ]:
package_payload = probe_workflow.package_probe_family_results(
    run_root=RUN_ROOT,
    family_root=FAMILY_ROOT,
    require_formal_pass_criteria=REQUIRE_FORMAL_PASS,
    formal_validation_summary=formal_validation_summary,
    drive_root=DRIVE_ROOT,
    family_id=FAMILY_ID,
    workflow_key=WORKFLOW_KEY,
    step_key=STEP_KEY,
    run_mode=RUN_MODE,
)
drive_archive_path = package_payload['drive_archive_path']
compat_pack_root = package_payload['compat_pack_root']
print({
    'drive_archive_path': drive_archive_path,
    'compat_pack_root': compat_pack_root,
    'result_registry.jsonl': package_payload.get('registry_paths', {}).get('result_registry.jsonl'),
    'family_registry.jsonl': package_payload.get('registry_paths', {}).get('family_registry.jsonl'),
})

### 15 Print Final Family Summary (`15_print_final_family_summary`)

作用：打印本次 family、run root、Drive package 和 formal validation summary，供人工核对与下游 handoff 使用。

In [ ]:
print({
    'family_id': FAMILY_ID,
    'run_root': str(RUN_ROOT),
    'family_root': str(FAMILY_ROOT),
    'formal_validation_summary': formal_validation_summary,
    'drive_archive_path': str(drive_archive_path),
})